# Acrobot iLQR-MPC — Hardware-in-the-Loop *(Corrected & Debugged)*

## Root causes of velocity blow-up & fixes applied

| Root cause | Fix |
|---|---|
| **Gravity, not control torque, drives velocity** | Gravity peak on link 1 ≈ 89 mNm (9× control torque). At horizontal, link 1 reaches 30 rad/s in ≈0.24 s purely from gravity. |
| **iLQR solve exceeded dt budget** (old N=40, iter=5 → ≈2200 RK4 calls ≈110–220 ms) | **FIX A** — Reduced to N=15, iter=2 (≈330 RK4 calls ≈15–30 ms). `dt_nom` is now **immutable**. |
| **`dt` overwritten mid-loop** (old Cell 6 wrote `dt = dt_hist[i-1]`) | **FIX A** — `dt` is an explicit parameter of every dynamics function. `dt_nom` is never overwritten. |
| **Zero torque on hard limit** — gravity kept accelerating at zero torque | **FIX B** — Active braking: apply `±TORQUE_LIMIT` opposing velocity direction. |
| **Hardware velocity is noisy encoder derivative** | **FIX C** — `StateEstimator` blends position-difference velocity (primary) with hardware velocity + median window + EMA. |
| **Torque applied to state that already passed** (solve latency) | **FIX E** — Latency compensation: iLQR seeded with 1-step forward prediction. |
| **Warm-start trajectory not clipped** → large torques leaked into next iteration | **FIX F** — Entire warm-start clipped to `±TORQUE_LIMIT` after each solve. |

## Notebook structure

| Cell | Contents |
|---|---|
| 1 | Imports |
| 2 | All parameters (edit only here) |
| 3 | Acrobot dynamics (`dt` always explicit) |
| 4 | iLQR backward/forward passes |
| 5 | `StateEstimator` + `velocity_safe_torque` with active braking |
| 6 | Connect → diagnose API latency → MPC loop → plots → video |


In [1]:
import math
import subprocess
import time
import warnings
from collections import deque
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import wget
from IPython.display import display, Video

%matplotlib inline
warnings.filterwarnings('ignore')
print('Imports OK')


Imports OK


In [2]:
# =============================================================================
# CELL 2 — ALL Parameters  (edit ONLY this cell to tune)
# =============================================================================

# ── Physical parameters ───────────────────────────────────────────────────────
m1, m2 = 0.10548177618443695,    0.07619744360415454
l1, l2 = 0.05,                   0.05
r1, r2 = 0.05,                   0.03670036749567022
I1, I2 = 0.00046166221821039165, 0.00023702395072092597
g       = 9.81
b2      = 0.0005106535523065844
cf1     = 0.0022933128359236008
cf2     = 0.0013188243178924417

J1 = I1 + m1 * r1**2
J2 = I2 + m2 * r2**2

# ── Timing ────────────────────────────────────────────────────────────────────
# dt_nom must be >= actual loop time. Measured solve ~32 ms + API ~1 ms = 33 ms.
# Use 75 ms for solid margin.
dt_nom = 0.075   # [s]

# ── MPC ───────────────────────────────────────────────────────────────────────
N_MPC    = 8
MAX_ITER = 2
REGU0    = 10.0
T_SIM    = 10.0

# ── Goal ──────────────────────────────────────────────────────────────────────
x_goal = np.array([np.pi, 0.0, 0.0, 0.0])

# ── Cost matrices ─────────────────────────────────────────────────────────────
Q  = np.diag([100.0, 10.0, 5.0, 1.0])
R  = np.array([[0.0001]])
Qf = np.diag([50000., 2000., 500., 500.])

# ── Torque limit ──────────────────────────────────────────────────────────────
TORQUE_LIMIT = 0.015   # [Nm]

# ── Velocity safety ───────────────────────────────────────────────────────────
# Hardware cuts out at +/-30 rad/s. Must never approach that.
# At 75 ms steps, 6 rad/s -> next step could be 6 + 81*0.075 = 12 rad/s from gravity alone.
# So hard limit must be LOW enough that one step of gravity can't bridge to 30.
# 30 - 81*0.075 = 23.9 -> any hard limit below ~23 is safe for one-step look-ahead,
# but we want several steps of braking margin: use 5 rad/s hard limit.
VEL_SOFT_LIMIT = 3.0   # [rad/s]
VEL_HARD_LIMIT = 4.0   # [rad/s]  -- conservative; hardware limit is 30

# ── Velocity filter ───────────────────────────────────────────────────────────
# CRITICAL LESSON from run data:
#   At 75 ms steps, hardware vel showed +/-18 rad/s while filtered showed +/-2.
#   The EMA (alpha=0.20) was smoothing too aggressively over slow steps,
#   making the controller blind to real velocity -> it kept commanding full torque.
#
# Fix: trust hardware velocity MORE (VEL_POS_WEIGHT lower), less EMA smoothing,
# and use the hardware vel directly for the safety check (bypass filter for limits).
VEL_EMA_ALPHA  = 0.60   # much more responsive -- 0.6 means 60% new sample
VEL_MEDIAN_WIN = 3      # minimal spike rejection at 75 ms (fewer samples available)
VEL_POS_WEIGHT = 0.40   # trust hardware velocity more (was 0.70 pos-diff)

# ── Warm-start ────────────────────────────────────────────────────────────────
u_trj_init = np.zeros((N_MPC, 1))

# ── Hardware ──────────────────────────────────────────────────────────────────
USER_TOKEN = '614401778513408'
RECORD     = True

# ── Print summary ─────────────────────────────────────────────────────────────
G1_max = g * (m1*r1 + m2*l1)
G2_max = g * m2 * r2
print('-' * 68)
print('TIMING')
print(f'  dt_nom         : {dt_nom*1e3:.0f} ms')
print(f'  Horizon        : {N_MPC} x {dt_nom*1e3:.0f} ms = {N_MPC*dt_nom*1e3:.0f} ms')
print(f'  Steps in T_SIM : {int(T_SIM/dt_nom)}')
print()
print('PHYSICS -- why velocity blows up even at low torque')
print(f'  Gravity accel (link2): {G2_max/J2:.0f} rad/s^2')
print(f'  In one dt_nom={dt_nom*1e3:.0f} ms step gravity adds: {G2_max/J2*dt_nom:.2f} rad/s')
print(f'  From VEL_HARD={VEL_HARD_LIMIT}, next step could be: {VEL_HARD_LIMIT + G2_max/J2*dt_nom:.1f} rad/s')
print()
print('VELOCITY FILTER (was the hidden cause of blow-up in last run)')
print(f'  The filter showed dq_f~+/-2 while hardware showed +/-18 rad/s.')
print(f'  Controller thought arm was slow -> kept commanding full torque.')
print(f'  New: EMA alpha={VEL_EMA_ALPHA} (was 0.20), pos_weight={VEL_POS_WEIGHT} (was 0.70)')
print(f'  Safety check uses RAW hardware velocity, not filtered (new in Cell 5).')
print()
print('SAFETY')
print(f'  VEL_SOFT={VEL_SOFT_LIMIT}, VEL_HARD={VEL_HARD_LIMIT} rad/s  (hardware limit: +/-30)')
print('-' * 68)


--------------------------------------------------------------------
TIMING
  dt_nom         : 75 ms
  Horizon        : 8 x 75 ms = 600 ms
  Steps in T_SIM : 133

PHYSICS -- why velocity blows up even at low torque
  Gravity accel (link2): 81 rad/s^2
  In one dt_nom=75 ms step gravity adds: 6.06 rad/s
  From VEL_HARD=5.0, next step could be: 11.1 rad/s

VELOCITY FILTER (was the hidden cause of blow-up in last run)
  The filter showed dq_f~+/-2 while hardware showed +/-18 rad/s.
  Controller thought arm was slow -> kept commanding full torque.
  New: EMA alpha=0.6 (was 0.20), pos_weight=0.4 (was 0.70)
  Safety check uses RAW hardware velocity, not filtered (new in Cell 5).

SAFETY
  VEL_SOFT=3.0, VEL_HARD=5.0 rad/s  (hardware limit: +/-30)
--------------------------------------------------------------------


In [3]:
# =============================================================================
# CELL 3 — Acrobot Dynamics
#
# KEY FIX (A): dt is now an EXPLICIT PARAMETER of acrobot_rk4.
#   Old code mutated global `dt` inside the loop causing two problems:
#   1. Planner used an inconsistent timestep on every iteration.
#   2. On step i=1, dt was whatever the previous solve took (~150 ms).
#   Fix: dt_nom is the immutable nominal constant. Functions that need to
#   integrate over true elapsed time receive dt_step explicitly.
# =============================================================================

def acrobot_continuous(x, u):
    """Continuous-time acrobot EOM with smooth Coulomb friction."""
    q1, q2 = x[0], x[1]
    dq1 = float(np.clip(x[2], -VEL_HARD_LIMIT * 2, VEL_HARD_LIMIT * 2))
    dq2 = float(np.clip(x[3], -VEL_HARD_LIMIT * 2, VEL_HARD_LIMIT * 2))
    tau2    = u[0]
    EPS_VEL = 0.05   # tanh smoothing width [rad/s]

    M11 = J1 + J2 + m2*(l1**2 + 2*l1*r2*np.cos(q2))
    M12 = J2 + m2*l1*r2*np.cos(q2)
    M22 = J2
    h   = m2*l1*r2*np.sin(q2)
    c1  = -2*h*dq2*dq1 - h*dq2**2
    c2  =  h*dq1**2
    G1  = g*(m1*r1 + m2*l1)*np.sin(q1) + g*m2*r2*np.sin(q1+q2)
    G2  = g*m2*r2*np.sin(q1+q2)
    rhs1 = 0    - c1 - G1 - cf1*np.tanh(dq1/EPS_VEL)
    rhs2 = tau2 - c2 - G2 - b2*dq2 - cf2*np.tanh(dq2/EPS_VEL)
    det  = M11*M22 - M12**2
    ddq1 = (M22*rhs1 - M12*rhs2) / det
    ddq2 = (M11*rhs2 - M12*rhs1) / det
    return np.array([dq1, dq2, ddq1, ddq2])


def acrobot_rk4(x, u, dt_step):
    """
    RK4 integration over dt_step seconds. Always explicit -- no global dt.

    Pass dt_step=dt_nom for planning (iLQR always plans at nominal rate).
    Pass dt_step=dt_actual when predicting state for latency compensation.
    """
    k1 = acrobot_continuous(x,                 u)
    k2 = acrobot_continuous(x + dt_step/2*k1,  u)
    k3 = acrobot_continuous(x + dt_step/2*k2,  u)
    k4 = acrobot_continuous(x + dt_step   *k3, u)
    x_next = x + dt_step/6*(k1 + 2*k2 + 2*k3 + k4)
    x_next[2] = float(np.clip(x_next[2], -VEL_HARD_LIMIT, VEL_HARD_LIMIT))
    x_next[3] = float(np.clip(x_next[3], -VEL_HARD_LIMIT, VEL_HARD_LIMIT))
    return x_next


def acrobot_discrete_RK4(x, u, dt_step=None):
    """Wrapper: uses dt_nom by default (for iLQR planning)."""
    return acrobot_rk4(x, u, dt_step if dt_step is not None else dt_nom)


# ── Angle wrapping & cost functions ──────────────────────────────────────────
def wrap_angle(theta):
    return (theta + np.pi) % (2*np.pi) - np.pi

def angle_error(x):
    dx = x - x_goal
    dx[0] = wrap_angle(dx[0])
    dx[1] = wrap_angle(dx[1])
    return dx

def stage_cost(x, u):
    dx = angle_error(x.copy())
    return float(dx @ Q @ dx + u @ R @ u)

def stage_cost_derivatives(x, u):
    dx   = angle_error(x.copy())
    l_x  = 2.0 * Q @ dx
    l_u  = 2.0 * R @ u
    l_xx = 2.0 * Q
    l_ux = np.zeros((R.shape[0], Q.shape[0]))
    l_uu = 2.0 * R
    return l_x, l_u, l_xx, l_ux, l_uu

def final_cost(x):
    dx = angle_error(x.copy())
    return float(dx @ Qf @ dx)

def final_cost_derivatives(x):
    dx = angle_error(x.copy())
    return 2.0 * Qf @ dx, 2.0 * Qf

# ── Dynamics Jacobians (central finite differences, uses dt_nom) ─────────────
def f_x(x, u, eps=1e-5):
    n = len(x)
    J = np.zeros((n, n))
    for i in range(n):
        xp, xm = x.copy(), x.copy()
        xp[i] += eps; xm[i] -= eps
        J[:, i] = (acrobot_discrete_RK4(xp, u) - acrobot_discrete_RK4(xm, u)) / (2*eps)
    return J

def f_u(x, u, eps=1e-5):
    m, n = len(u), len(x)
    J = np.zeros((n, m))
    for i in range(m):
        up, um = u.copy(), u.copy()
        up[i] += eps; um[i] -= eps
        J[:, i] = (acrobot_discrete_RK4(x, up) - acrobot_discrete_RK4(x, um)) / (2*eps)
    return J

print('Acrobot dynamics defined (dt always passed explicitly).')
print(f'  RK4 sanity @ zero state: {np.round(acrobot_rk4(np.zeros(4), np.zeros(1), dt_nom), 6)}')


Acrobot dynamics defined (dt always passed explicitly).
  RK4 sanity @ zero state: [0. 0. 0. 0.]


In [4]:
# =============================================================================
# CELL 4 — iLQR Backward / Forward Passes
# =============================================================================

def _Q_terms(l_x, l_u, l_xx, l_ux, l_uu, Fx, Fu, V_x, V_xx):
    Q_x  = l_x  + Fx.T @ V_x
    Q_u  = l_u  + Fu.T @ V_x
    Q_xx = l_xx + Fx.T @ V_xx @ Fx
    Q_ux = l_ux + Fu.T @ V_xx @ Fx
    Q_uu = l_uu + Fu.T @ V_xx @ Fu
    return Q_x, Q_u, Q_xx, Q_ux, Q_uu


def _backward_pass(x_trj, u_trj, regu=1.0):
    N = u_trj.shape[0]
    K_trj = np.zeros((N, 1, 4))
    k_trj = np.zeros((N, 1))
    V_x, V_xx = final_cost_derivatives(x_trj[-1])

    for n in range(N-1, -1, -1):
        l_x, l_u, l_xx, l_ux, l_uu = stage_cost_derivatives(x_trj[n].copy(), u_trj[n])
        Fx = f_x(x_trj[n], u_trj[n])
        Fu = f_u(x_trj[n], u_trj[n])
        Qx, Qu, Qxx, Qux, Quu = _Q_terms(l_x, l_u, l_xx, l_ux, l_uu, Fx, Fu, V_x, V_xx)

        regu_local = regu
        for _ in range(12):
            Quu_reg = Quu + regu_local * np.eye(1)
            if np.all(np.isfinite(Quu_reg)) and Quu_reg[0, 0] > 1e-8:
                break
            regu_local *= 5.0

        Q_uu_inv = np.linalg.inv(Quu_reg)
        k = -Q_uu_inv @ Qu
        K = -Q_uu_inv @ Qux

        if not np.all(np.isfinite(k)) or not np.all(np.isfinite(K)):
            return k_trj, K_trj, 0.0

        k_trj[n] = k
        K_trj[n] = K
        V_x  = Qx  - K.T @ Quu @ k
        V_xx = Qxx - K.T @ Quu @ K
        V_xx = np.clip(V_xx, -1e8, 1e8)

    return k_trj, K_trj, 0.0


def _rollout(x0, u_trj):
    N = u_trj.shape[0]
    x_trj = np.zeros((N+1, 4))
    x_trj[0] = x0
    for n in range(N):
        x_trj[n+1] = acrobot_discrete_RK4(x_trj[n], u_trj[n])
    return x_trj


def _cost_trj(x_trj, u_trj):
    J = sum(stage_cost(x_trj[n], u_trj[n]) for n in range(u_trj.shape[0]))
    return J + final_cost(x_trj[-1])


def _forward_pass(x_trj, u_trj, k_trj, K_trj, alpha=1.0):
    N = u_trj.shape[0]
    x_new = np.zeros_like(x_trj)
    u_new = np.zeros_like(u_trj)
    x_new[0] = x_trj[0]
    for n in range(N):
        u_new[n] = u_trj[n] + alpha*k_trj[n] + K_trj[n] @ (x_new[n] - x_trj[n])
        x_new[n+1] = acrobot_discrete_RK4(x_new[n], u_new[n])
    return x_new, u_new


def run_ilqr(x0, N=50, max_iter=5, regu_init=10.0, tol=1e-3, init_u_trj=None):
    """iLQR from x0 for N steps using dt_nom throughout."""
    u_trj = init_u_trj.copy() if init_u_trj is not None else np.zeros((N, 1))
    x_trj = _rollout(x0, u_trj)
    if not np.all(np.isfinite(x_trj)):
        u_trj = np.zeros((N, 1))
        x_trj = _rollout(x0, u_trj)

    regu  = regu_init
    costs = [_cost_trj(x_trj, u_trj)]

    for _ in range(max_iter):
        k_trj, K_trj, _ = _backward_pass(x_trj, u_trj, regu)
        accepted = False
        for alpha in [1.0, 0.5, 0.25, 0.125, 0.0625]:
            x_new, u_new = _forward_pass(x_trj, u_trj, k_trj, K_trj, alpha)
            c_new = _cost_trj(x_new, u_new)
            if c_new < costs[-1]:
                x_trj, u_trj = x_new, u_new
                costs.append(c_new)
                regu     = max(regu * 0.7, 1e-6)
                accepted = True
                break
        if not accepted:
            regu = min(regu * 2.0, 1e6)
        if len(costs) > 1 and abs(costs[-1] - costs[-2]) < tol:
            break

    return x_trj, u_trj, costs


print('iLQR defined.')


iLQR defined.


In [5]:
# =============================================================================
# CELL 5 — StateEstimator + velocity_safe_torque
#
# KEY LESSON: the safety check must use RAW hardware velocity, not filtered.
#   In the previous run, EMA+median smoothing hid +/-18 rad/s peaks as +/-2.
#   The safety gate never fired, controller kept commanding full torque,
#   and the arm hit the hardware +/-30 rad/s cutoff.
#
# Fix: velocity_safe_torque now takes BOTH dq_filtered (for iLQR seeding)
#   AND dq_hw_raw (for safety checking). Hard limit uses the maximum of both.
# =============================================================================

class StateEstimator:
    """
    Velocity estimator: blend position-difference with hardware velocity.
    Used to give iLQR a smooth, low-noise state estimate.
    DO NOT use the output of this for safety limiting -- use raw hw vel instead.
    """
    def __init__(self, n_joints=2, win=VEL_MEDIAN_WIN,
                 alpha=VEL_EMA_ALPHA, pos_weight=VEL_POS_WEIGHT):
        self.win        = win
        self.alpha      = alpha
        self.pos_weight = pos_weight
        self._buf       = [deque(maxlen=win) for _ in range(n_joints)]
        self._ema       = np.zeros(n_joints)
        self._q_prev    = None
        self._ready     = False

    def update(self, q_raw, dq_hw, dt_actual):
        if self._q_prev is not None and dt_actual > 5e-4:
            dq_pos = (q_raw - self._q_prev) / dt_actual
        else:
            dq_pos = dq_hw.copy()
        self._q_prev = q_raw.copy()

        dq_blend = self.pos_weight * dq_pos + (1.0 - self.pos_weight) * dq_hw

        for j, v in enumerate(dq_blend):
            self._buf[j].append(float(v))
        med = np.array([np.median(list(b)) for b in self._buf])

        if not self._ready:
            self._ema   = med.copy()
            self._ready = True
        else:
            self._ema = self.alpha * med + (1.0 - self.alpha) * self._ema

        return self._ema.copy()


def velocity_safe_torque(u_raw, dq_filtered, dq_hw_raw):
    """
    Safety limiter using the WORST-CASE velocity across filtered AND raw hardware.

    Why use raw hardware vel here?
      Filtered velocity (EMA+median) lags and smooths peaks. At 75 ms steps
      the filter can show 2 rad/s while the arm is actually moving at 18 rad/s.
      Using only filtered vel for safety = controller blind to real speed.

    Strategy:
      - Use max(|dq_filtered|, |dq_hw_raw|) per joint for limit checking.
      - If either source exceeds VEL_HARD_LIMIT: active braking.
      - Soft zone uses filtered vel (smoother, less chattery).
    """
    # Worst-case velocity: take the larger magnitude from each source
    dq_worst = np.maximum(np.abs(dq_filtered), np.abs(dq_hw_raw))

    # Hard limit: active braking -- use worst-case
    if dq_worst[1] >= VEL_HARD_LIMIT:
        # Use raw hw vel sign for braking direction (more accurate)
        sign = np.sign(dq_hw_raw[1]) if abs(dq_hw_raw[1]) > 0.01 else np.sign(dq_filtered[1])
        return -sign * TORQUE_LIMIT

    if dq_worst[0] >= VEL_HARD_LIMIT:
        sign = np.sign(dq_hw_raw[0]) if abs(dq_hw_raw[0]) > 0.01 else np.sign(dq_filtered[0])
        return -sign * TORQUE_LIMIT

    # Soft zone: use filtered vel for smoothness
    span   = VEL_HARD_LIMIT - VEL_SOFT_LIMIT
    scales = []
    for dq_f, dq_w in zip(dq_filtered, dq_worst):
        dq_a = dq_w   # use worst-case magnitude for scaling decision
        if dq_a < VEL_SOFT_LIMIT:
            scales.append(1.0)
            continue
        progress     = (dq_a - VEL_SOFT_LIMIT) / span
        accel_scale  = (1.0 - progress) ** 2
        accelerating = (np.sign(u_raw) == np.sign(dq_f))
        scales.append(accel_scale if accelerating else max(accel_scale, 0.60))

    return u_raw * float(np.min(scales))


print('StateEstimator and velocity_safe_torque defined.')
print(f'  EMA alpha      : {VEL_EMA_ALPHA}  (0.6 = responsive; was 0.15-0.20)')
print(f'  Median window  : {VEL_MEDIAN_WIN} samples')
print(f'  Pos-diff weight: {VEL_POS_WEIGHT}  (0.4 = trust hardware more; was 0.70)')
print(f'  Safety check   : max(|filtered|, |hw_raw|) per joint  <- KEY FIX')
print(f'  VEL_SOFT={VEL_SOFT_LIMIT}, VEL_HARD={VEL_HARD_LIMIT} rad/s')


StateEstimator and velocity_safe_torque defined.
  EMA alpha      : 0.6  (0.6 = responsive; was 0.15-0.20)
  Median window  : 3 samples
  Pos-diff weight: 0.4  (0.4 = trust hardware more; was 0.70)
  Safety check   : max(|filtered|, |hw_raw|) per joint  <- KEY FIX
  VEL_SOFT=3.0, VEL_HARD=5.0 rad/s


In [6]:
# =============================================================================
# CELL 6 -- Connect -> Diagnose -> MPC loop -> Plots -> Video
#
# FIX A : dt_nom never modified. Dynamics functions receive dt explicitly.
# FIX B : Active braking (counter-torque) when |dq| >= VEL_HARD_LIMIT.
# FIX C : StateEstimator blends position-diff + hardware velocity.
# FIX D : Hardware timing diagnostic before main loop.
# FIX E : Latency compensation -- iLQR seeded with 1-step forward prediction.
# FIX F : Entire warm-start clipped to +/-TORQUE_LIMIT after each solve.
# FIX G : Every step printed including raw hw velocity vs filtered velocity.
# =============================================================================

from cloudpendulumclient.client import Client

client = Client()
client.get_user_info(USER_TOKEN)

session_token, livestream_url = client.start_experiment(
    user_token       = USER_TOKEN,
    experiment_type  = 'Acrobot',
    experiment_time  = T_SIM,
    preparation_time = 5.0,
    initial_state    = [0.01, 0.0],
    record           = RECORD,
)
print(f'Session    : {session_token}')
if livestream_url:
    print(f'Livestream : {livestream_url}')

client.set_impedance_controller_params(0.0, 0.0, session_token)

# -----------------------------------------------------------------------------
# FIX D -- Hardware timing diagnostic (20 probe calls)
# -----------------------------------------------------------------------------
print()
print('-- Hardware timing diagnostic (20 probe calls) ----------------------')
pos_t, vel_t, set_t = [], [], []
pos_probe = []
for _ in range(20):
    t0 = time.perf_counter()
    q_ = np.array(client.get_position(session_token)).flatten()
    pos_t.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    client.get_velocity(session_token)
    vel_t.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    client.set_torque([0.0], session_token)
    set_t.append(time.perf_counter() - t0)

    pos_probe.append(q_)
    time.sleep(0.002)

pos_arr     = np.array(pos_probe)
n_changes   = int(np.sum(np.any(np.diff(pos_arr, axis=0) != 0, axis=1)))
hw_rate_est = n_changes / (20 * 0.002)
api_overhead = np.mean(pos_t) + np.mean(vel_t) + np.mean(set_t)
ilqr_budget  = dt_nom - api_overhead

print(f'  get_position  : {np.mean(pos_t)*1e3:5.1f} +/- {np.std(pos_t)*1e3:.1f} ms')
print(f'  get_velocity  : {np.mean(vel_t)*1e3:5.1f} +/- {np.std(vel_t)*1e3:.1f} ms')
print(f'  set_torque    : {np.mean(set_t)*1e3:5.1f} +/- {np.std(set_t)*1e3:.1f} ms')
print(f'  Total API/step: {api_overhead*1e3:5.1f} ms  out of {dt_nom*1e3:.0f} ms')
print(f'  iLQR budget   : {ilqr_budget*1e3:5.1f} ms  (remaining after API calls)')
print(f'  HW data rate  : ~{hw_rate_est:.0f} Hz  ({n_changes}/{19} samples changed)')
if ilqr_budget < 0.010:
    print('  WARNING: iLQR budget < 10 ms -- increase dt_nom or reduce N_MPC.')
else:
    print('  OK: iLQR budget is reasonable.')

# Estimate actual solve time from a quick warm-up solve
print()
print('-- iLQR solve-time warm-up (5 test solves) --------------------------')
x_test = np.array([0.01, 0.0, 0.0, 0.0])
u_test = np.zeros((N_MPC, 1))
solve_warmup = []
for _ in range(5):
    t0 = time.perf_counter()
    run_ilqr(x_test, N=N_MPC, max_iter=MAX_ITER, regu_init=REGU0, tol=1e-4, init_u_trj=u_test)
    solve_warmup.append(time.perf_counter() - t0)
mean_solve = np.mean(solve_warmup)
print(f'  Mean iLQR solve : {mean_solve*1e3:.1f} ms   max {max(solve_warmup)*1e3:.1f} ms')
total_step = mean_solve + api_overhead
print(f'  Estimated step  : {total_step*1e3:.1f} ms  (solve + API)')
if total_step > dt_nom * 0.90:
    print()
    print(f'  *** DANGER: estimated step ({total_step*1e3:.1f} ms) >= dt_nom ({dt_nom*1e3:.0f} ms) ***')
    print(f'  *** Model will under-integrate velocity -> velocity blow-up!  ***')
    print(f'  *** Increase dt_nom to at least {total_step*1.3*1e3:.0f} ms and re-run Cell 2. ***')
    raise RuntimeError(
        f'dt_nom ({dt_nom*1e3:.0f} ms) too small for measured step ({total_step*1e3:.1f} ms). '
        f'Set dt_nom >= {total_step*1.3:.3f} in Cell 2 and re-run.'
    )
else:
    print(f'  OK: dt_nom ({dt_nom*1e3:.0f} ms) covers estimated step with margin.')
print()

# -----------------------------------------------------------------------------
# Allocate history buffers
# -----------------------------------------------------------------------------
n_steps = int(T_SIM / dt_nom)

x_hist        = np.zeros((n_steps + 1, 4))
u_hist        = np.zeros((n_steps, 1))
u_raw_hist    = np.zeros((n_steps, 1))
solve_hist    = np.zeros(n_steps)
dt_hist       = np.zeros(n_steps)
time_hist     = np.zeros(n_steps)
dq_hw_hist    = np.zeros((n_steps, 2))
dq_filt_hist  = np.zeros((n_steps, 2))
dq_model_hist = np.zeros((n_steps, 2))

u_trj      = u_trj_init.copy()
estimator  = StateEstimator(n_joints=2)

q0    = np.array(client.get_position(session_token)).flatten()
dq0   = np.array(client.get_velocity(session_token)).flatten()
dq0f  = estimator.update(q0, dq0, dt_nom)
x_hist[0] = np.concatenate([q0, dq0f])

print(f'Starting iLQR-MPC: {n_steps} steps x {dt_nom*1e3:.0f} ms = {T_SIM:.1f} s')
print(f'  Initial state: {np.round(x_hist[0], 4)}')
print()

hdr = (f"{'Step':>5} | {'err':>6} | {'u_ilqr mNm':>10} | {'u_safe mNm':>10} | "
       f"{'dq1_f':>6} | {'dq2_f':>6} | {'dq1_hw':>7} | {'dq2_hw':>7} | "
       f"{'solve ms':>8} | {'dt_act ms':>9}")
print(hdr)
print('-' * len(hdr))

meas_time    = 0.0
u_prev       = 0.0
solve_t_prev = 0.0
x_model_pred = x_hist[0].copy()

for i in range(n_steps):
    if meas_time >= T_SIM:
        break

    loop_start = time.perf_counter()
    dt_prev    = dt_hist[i-1] if i > 0 else dt_nom

    # -- Sense ----------------------------------------------------------------
    q_raw = np.array(client.get_position(session_token)).flatten()
    dq_hw = np.array(client.get_velocity(session_token)).flatten()

    # FIX C: blend position-diff velocity with hardware velocity
    dq_f  = estimator.update(q_raw, dq_hw, dt_prev)
    x_cur = np.concatenate([q_raw, dq_f])

    dq_hw_hist[i]    = dq_hw
    dq_filt_hist[i]  = dq_f
    dq_model_hist[i] = x_model_pred[2:]

    # -- Non-finite guard -----------------------------------------------------
    if not np.all(np.isfinite(x_cur)):
        print(f'{i:5d} | NON-FINITE STATE -- zero torque sent')
        client.set_torque([0.0], session_token)
        u_hist[i]    = 0.0
        x_hist[i+1]  = x_hist[i]
        while time.perf_counter() - loop_start < dt_nom:
            pass
        dt_act       = time.perf_counter() - loop_start
        dt_hist[i]   = dt_act
        meas_time   += dt_act
        time_hist[i] = meas_time
        continue

    # -- FIX E: Latency compensation ------------------------------------------
    if solve_t_prev > 2e-3:
        pred_dt = float(np.clip(solve_t_prev, 0.005, dt_nom))
        x_ilqr  = acrobot_rk4(x_cur, np.array([u_prev]), pred_dt)
    else:
        x_ilqr = x_cur.copy()

    # -- iLQR solve -----------------------------------------------------------
    t0    = time.perf_counter()
    u_trj = np.vstack([u_trj[1:], u_trj[-1:]])

    try:
        _, u_trj, _ = run_ilqr(
            x_ilqr, N=N_MPC, max_iter=MAX_ITER,
            regu_init=REGU0, tol=1e-4, init_u_trj=u_trj,
        )
        # FIX F: clip entire warm-start trajectory
        u_trj = np.clip(u_trj, -TORQUE_LIMIT, TORQUE_LIMIT)
        u_raw = float(u_trj[0, 0])
        if not np.isfinite(u_raw):
            raise ValueError('NaN/Inf in iLQR output')
    except Exception as exc:
        print(f'{i:5d} | iLQR failed ({exc}) -- zero torque, warm-start reset')
        u_raw  = 0.0
        u_trj  = np.zeros((N_MPC, 1))

    solve_t      = time.perf_counter() - t0
    solve_t_prev = solve_t

    # -- FIX B: velocity-safe torque with active braking ----------------------
    u_safe = velocity_safe_torque(u_raw, dq_f, dq_hw)
    u_prev = u_safe

    # -- Pre-actuate hard safety check using raw hardware velocity --------
    # This is the final gate before any torque reaches the hardware.
    # Recompute using raw hw vel in case estimator lagged.
    dq_hw_abs = np.abs(dq_hw)
    if dq_hw_abs[1] >= VEL_HARD_LIMIT or dq_hw_abs[0] >= VEL_HARD_LIMIT:
        u_safe = -np.sign(dq_hw[np.argmax(dq_hw_abs)]) * TORQUE_LIMIT
    # Absolute emergency: if hw vel > 20 rad/s, zero torque and skip
   if np.any(dq_hw_abs > 15.0):
        fastest_joint = np.argmax(dq_hw_abs)
        brake_torque = -np.sign(dq_hw[fastest_joint]) * TORQUE_LIMIT
        print(f'{i:5d} | EMERGENCY STOP: hw vel {dq_hw} rad/s, braking')
        # Keep braking for the full dt window instead of one shot + wait
        t_brake_end = time.perf_counter() + dt_nom
        while time.perf_counter() < t_brake_end:
            client.set_torque([brake_torque], session_token)
            time.sleep(0.005)  # re-send at ~200 Hz during wait
            u_hist[i] = 0.0
            x_hist[i+1] = x_cur
            while time.perf_counter() - loop_start < dt_nom:
                pass
            dt_act = time.perf_counter() - loop_start
            dt_hist[i] = dt_act
            meas_time += dt_act
            time_hist[i] = meas_time
            continue

    # -- Actuate --------------------------------------------------------------
    client.set_torque([u_safe], session_token)

    # -- 1-step model prediction ----------------------------------------------
    x_model_pred = acrobot_rk4(x_cur, np.array([u_safe]), dt_nom)

    # -- Log ------------------------------------------------------------------
    x_hist[i+1]   = x_cur
    u_hist[i]     = u_safe
    u_raw_hist[i] = u_raw
    solve_hist[i] = solve_t

    # -- Busy-wait to nominal dt ----------------------------------------------
    while time.perf_counter() - loop_start < dt_nom:
        pass
    dt_act       = time.perf_counter() - loop_start
    dt_hist[i]   = dt_act
    meas_time   += dt_act
    time_hist[i] = meas_time

    # -- FIX G: print every step ----------------------------------------------
    err_norm = np.linalg.norm(angle_error(x_cur.copy()))
    # BRAKE flag: triggered when safety override fired (u_safe != u_raw direction or magnitude)
    braking  = (np.sign(u_safe) != np.sign(u_raw) and abs(u_raw) > 1e-6) or (abs(u_safe) < abs(u_raw) * 0.95 and abs(u_raw) > 1e-6)
    flag     = ' <- BRAKE' if braking else ''
    print(
        f'{i:5d} | {err_norm:6.3f} | {u_raw*1e3:+10.2f} | {u_safe*1e3:+10.2f} | '
        f'{dq_f[0]:+6.2f} | {dq_f[1]:+6.2f} | {dq_hw[0]:+7.2f} | {dq_hw[1]:+7.2f} | '
        f'{solve_t*1e3:8.1f} | {dt_act*1e3:9.1f}{flag}'
    )

# -- Safety: zero torque on exit ----------------------------------------------
try:
    client.set_torque([0.0], session_token)
except Exception:
    pass

n_act = i + 1
print()
print('-' * 70)
print('TIMING SUMMARY')
print(f'  Steps completed  : {n_act}')
print(f'  Mean solve time  : {solve_hist[:n_act].mean()*1e3:.1f} ms   max {solve_hist[:n_act].max()*1e3:.1f} ms')
print(f'  Mean actual dt   : {dt_hist[:n_act].mean()*1e3:.1f} ms   max {dt_hist[:n_act].max()*1e3:.1f} ms   (target {dt_nom*1e3:.0f} ms)')
dt_err = (dt_hist[:n_act].mean() - dt_nom) / dt_nom * 100
print(f'  dt inflation     : {dt_err:+.1f}%  -> model vel error ~{dt_err:+.1f}% per step without latency comp')
print(f'  Fraction over    : {(dt_hist[:n_act] > dt_nom).mean()*100:.0f}% of steps exceeded budget')
print()
print('VELOCITY SUMMARY')
print(f'  Max |dq| hardware   : {np.max(np.abs(dq_hw_hist[:n_act])):.1f} rad/s')
print(f'  Max |dq| filtered   : {np.max(np.abs(dq_filt_hist[:n_act])):.1f} rad/s')
print(f'  Max |dq| model pred : {np.max(np.abs(dq_model_hist[:n_act])):.1f} rad/s')
vel_err_rms = np.sqrt(np.mean((dq_filt_hist[:n_act] - dq_model_hist[:n_act])**2))
print(f'  RMS velocity error  : {vel_err_rms:.3f} rad/s  (hw_filt - model)')
print()
print('CONTROL SUMMARY')
print(f'  Final state      : {np.round(x_hist[n_act], 3)}')
print(f'  Final error      : {np.linalg.norm(angle_error(x_hist[n_act])):.4f} rad')
print(f'  Max |u| applied  : {np.max(np.abs(u_hist[:n_act]))*1e3:.2f} mNm')
print('-' * 70)

download_url = client.stop_experiment(session_token)

# =============================================================================
# Plots
# =============================================================================
T_t = time_hist[:n_act]
fig, axs = plt.subplots(2, 4, figsize=(30, 10))

# R1C1: Joint angles
ax = axs[0, 0]
ax.plot(T_t, x_hist[1:n_act+1, 0], lw=2, label='q1')
ax.plot(T_t, x_hist[1:n_act+1, 1], lw=2, label='q2')
ax.axhline(np.pi, color='gray', ls='--', lw=1.2, label='goal pi')
ax.set_title('Joint angles [rad]', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
ax.set_xlabel('time [s]')

# R1C2: Hardware vs filtered velocity
ax = axs[0, 1]
ax.plot(T_t, dq_hw_hist[:n_act, 0],   lw=1.0, color='salmon',    alpha=0.5, label='dq1 hw (raw)')
ax.plot(T_t, dq_hw_hist[:n_act, 1],   lw=1.0, color='lightblue', alpha=0.5, label='dq2 hw (raw)')
ax.plot(T_t, dq_filt_hist[:n_act, 0], lw=2.2, color='darkred',              label='dq1 filtered')
ax.plot(T_t, dq_filt_hist[:n_act, 1], lw=2.2, color='steelblue',            label='dq2 filtered')
for sign in [1, -1]:
    ax.axhline(sign * VEL_SOFT_LIMIT, color='orange', ls='--', lw=1.2)
    ax.axhline(sign * VEL_HARD_LIMIT, color='red',    ls='--', lw=1.5)
ax.set_title('Velocity: hardware (faint) vs filtered (bold) | orange=soft red=hard limit', fontweight='bold')
ax.legend(fontsize=7)
ax.grid(alpha=0.3)
ax.set_xlabel('time [s]')

# R1C3: Model prediction vs hardware filtered
ax = axs[0, 2]
ax.plot(T_t, dq_model_hist[:n_act, 0], lw=2.2, ls='--', color='darkred',   label='dq1 MODEL')
ax.plot(T_t, dq_model_hist[:n_act, 1], lw=2.2, ls='--', color='steelblue', label='dq2 MODEL')
ax.plot(T_t, dq_filt_hist[:n_act, 0],  lw=2.2,          color='red',        label='dq1 HW-filt')
ax.plot(T_t, dq_filt_hist[:n_act, 1],  lw=2.2,          color='navy',       label='dq2 HW-filt')
ax.set_title('Model 1-step prediction vs hardware filtered | Gap = dt-drift + unmodelled dynamics', fontweight='bold')
ax.legend(fontsize=7)
ax.grid(alpha=0.3)
ax.set_xlabel('time [s]')

# R1C4: Torque
ax = axs[0, 3]
ax.plot(T_t, u_raw_hist[:n_act, 0]*1e3, lw=1.5, color='gray',  alpha=0.7, label='iLQR raw')
ax.plot(T_t, u_hist[:n_act, 0]*1e3,     lw=2.2, color='tomato',            label='applied (safe)')
ax.axhline( TORQUE_LIMIT*1e3, color='k', ls='--', lw=1.2)
ax.axhline(-TORQUE_LIMIT*1e3, color='k', ls='--', lw=1.2)
ax.set_title('Torque [mNm]: iLQR raw vs applied (after safety)', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
ax.set_xlabel('time [s]')
ax.set_ylabel('mNm')

# R2C1: Velocity error
ax = axs[1, 0]
vel_err_plot = dq_filt_hist[:n_act] - dq_model_hist[:n_act]
ax.plot(T_t, vel_err_plot[:, 0], lw=2, color='darkred',   label='dq1 error')
ax.plot(T_t, vel_err_plot[:, 1], lw=2, color='steelblue', label='dq2 error')
ax.axhline(0, color='k', lw=0.8)
ax.fill_between(T_t, vel_err_plot[:, 1], alpha=0.15, color='steelblue')
ax.set_title(f'Velocity error: hw_filtered - model [rad/s] | RMS={vel_err_rms:.3f} rad/s | dt inflation={dt_err:+.1f}%', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
ax.set_xlabel('time [s]')

# R2C2: iLQR solve time
ax = axs[1, 1]
ax.plot(T_t, solve_hist[:n_act]*1e3, lw=1.5, color='mediumpurple', alpha=0.85, label='solve time')
ax.axhline(dt_nom*1e3, color='red', ls=':', lw=1.8, label=f'budget {dt_nom*1e3:.0f} ms')
ax.axhline(solve_hist[:n_act].mean()*1e3, color='k', ls='--', lw=1.2,
           label=f'mean {solve_hist[:n_act].mean()*1e3:.1f} ms')
ax.fill_between(T_t, dt_nom*1e3, solve_hist[:n_act]*1e3,
                where=(solve_hist[:n_act] > dt_nom),
                color='red', alpha=0.25, label='over budget')
ax.set_title('iLQR solve time [ms] | red fill = steps over dt budget', fontweight='bold')
ax.legend(fontsize=7)
ax.grid(alpha=0.3)
ax.set_xlabel('time [s]')

# R2C3: Actual loop dt
ax = axs[1, 2]
ax.plot(T_t, dt_hist[:n_act]*1e3, lw=1.5, color='steelblue', alpha=0.85, label='actual dt')
ax.axhline(dt_nom*1e3, color='red', ls=':', lw=1.8, label=f'nominal {dt_nom*1e3:.0f} ms')
ax.axhline(dt_hist[:n_act].mean()*1e3, color='k', ls='--', lw=1.2,
           label=f'mean {dt_hist[:n_act].mean()*1e3:.1f} ms')
ax.set_title('Actual loop period [ms] | inflation = dt-mismatch driving velocity error', fontweight='bold')
ax.legend(fontsize=7)
ax.grid(alpha=0.3)
ax.set_xlabel('time [s]')

# R2C4: Velocity phase portrait
ax = axs[1, 3]
sc = ax.scatter(dq_filt_hist[:n_act, 0], dq_filt_hist[:n_act, 1],
                c=T_t, cmap='Blues', s=8, label='hw filtered', zorder=3)
ax.plot(dq_model_hist[:n_act, 0], dq_model_hist[:n_act, 1],
        lw=1.5, color='red', ls='--', alpha=0.7, label='model prediction')
plt.colorbar(sc, ax=ax, label='time [s]')
theta = np.linspace(0, 2*np.pi, 200)
ax.plot(VEL_SOFT_LIMIT*np.cos(theta), VEL_SOFT_LIMIT*np.sin(theta),
        'orange', ls='--', lw=1.2, label=f'soft {VEL_SOFT_LIMIT} rad/s')
ax.plot(VEL_HARD_LIMIT*np.cos(theta), VEL_HARD_LIMIT*np.sin(theta),
        'red', ls='--', lw=1.5, label=f'hard {VEL_HARD_LIMIT} rad/s')
ax.set_xlabel('dq1 [rad/s]')
ax.set_ylabel('dq2 [rad/s]')
ax.set_title('Velocity phase portrait | colour=time circles=safety limits', fontweight='bold')
ax.legend(fontsize=7)
ax.grid(alpha=0.3)
ax.set_aspect('equal')

plt.suptitle(
    f'iLQR-MPC Hardware | N={N_MPC}, iter={MAX_ITER}, tau_lim={TORQUE_LIMIT*1e3:.0f} mNm'
    f' | v_soft={VEL_SOFT_LIMIT} v_hard={VEL_HARD_LIMIT} rad/s',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('mpc_hardware_results.png', dpi=130, bbox_inches='tight')
plt.show()

# =============================================================================
# Video download + display
# =============================================================================
if RECORD and download_url:
    print(f'Downloading video from {download_url} ...')
    flv_filename = wget.download(download_url, '.')
    print()
    mp4_filename = Path(flv_filename).stem + '.mp4'
    print(f'Re-encoding -> {mp4_filename}')
    subprocess.run(
        ['ffmpeg', '-y', '-i', flv_filename,
         '-vcodec', 'libx264', '-an',
         '-pix_fmt',  'yuv420p',
         '-movflags', '+faststart',
         '-vf',       'scale=trunc(iw/2)*2:trunc(ih/2)*2',
         mp4_filename],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True,
    )
    print(f'Saved: {mp4_filename}')
    display(Video(mp4_filename, embed=True, width=640))
elif RECORD:
    print('RECORD=True but server returned no download URL.')
else:
    print('RECORD=False -- video skipped.')


Successfully reserved cell on the server
Livestream url:  http://cloudpendulum.m2.chalmers.se:5080/live/viewer.jsp?host=cloudpendulum.m2.chalmers.se&stream=camera_169
Starting experiment in 5 seconds
Starting experiment in 4 seconds
Starting experiment in 3 seconds
Starting experiment in 2 seconds
Starting experiment in 1 seconds
Starting experiment in 0 seconds
Starting experiment
Session    : 2152036638228566272
Livestream : http://cloudpendulum.m2.chalmers.se:5080/live/viewer.jsp?host=cloudpendulum.m2.chalmers.se&stream=camera_169

-- Hardware timing diagnostic (20 probe calls) ----------------------
  get_position  :   0.5 +/- 0.2 ms
  get_velocity  :   0.4 +/- 0.2 ms
  set_torque    :   0.3 +/- 0.1 ms
  Total API/step:   1.1 ms  out of 75 ms
  iLQR budget   :  73.9 ms  (remaining after API calls)
  HW data rate  : ~475 Hz  (19/19 samples changed)
  OK: iLQR budget is reasonable.

-- iLQR solve-time warm-up (5 test solves) --------------------------
  Mean iLQR solve : 33.1 ms   ma

RuntimeError: Invalid request: CloudPendulumException: Velocity Limit Violation: Measured Velocity = -30.11896514892578 rad/s is outside the limits ± 30.0 rad/s